### Importing Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

### Loading the Dataset

In [2]:
df = pd.read_csv("fau_transline_recommender.csv")

### Cleaning the data set

In [107]:
df['personality_traits'] = df['personality_traits'].apply(lambda x: ' '.join([trait.strip() for trait in x.split(',')]))
df['hobbies'] = df['hobbies'].str.replace(',', '', regex=False)

### Creating a Textual Soup

Combines all relevant textual features (`teams`, `previous_experience`, `hobbies`, `sports`, `personality_traits`) into a single string for each employee to enable similarity computation.

In [108]:
def create_soup(x):
    return ' '.join([
        x['teams'],
        x['previous_experience'],
        x['hobbies'],
        x['sports'],
        x['personality_traits']
    ])
df['soup'] = df.apply(create_soup, axis=1)      

### Calculating Tf-Idf for Cosine Similarity
We apply TfidfVectorizer to convert the combined employee feature text (soup) into numerical vectors. This transforms each employee's profile into a format suitable for similarity comparison, while ignoring common English stop words. The resulting tfidf_matrix contains the TF-IDF scores for each term across all employee profiles

In [109]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['soup'])
tfidf_matrix.shape

(50, 58)

### Recommending Top 3 Employees using Cosine Similarity

In [110]:
def get_recommendations(id, indices, cosine_similarity=cosine_similarity):
    # get the index of the employee that matches the employee ID
    idx = indices[id]
    
    # get the pairwise similarity scores of all employees with the specified employee ID
    cosine_similarity_score = list(enumerate(cosine_similarity[idx]))
    
    # sort based on the similarity scores
    cosine_similarity_score = sorted(cosine_similarity_score, key=lambda x: x[1], reverse=True)
        
    # get indices of the three most similar items
    indices = [i[0] for i in cosine_similarity_score[1:4]]

    return indices

In [111]:
cosine_similarity = linear_kernel(tfidf_matrix, tfidf_matrix)

# construct a map of indices and ids
indices = pd.Series(df.index, index=df['id']).drop_duplicates()

recommended_indices = get_recommendations('klara_007', indices, cosine_similarity)

### Displaying the Results

In [112]:
print("Profile of klara_007")
print("-" * 130)
print(df[df['id'] == 'klara_007'].drop(columns=['soup']).to_string(index=False))
print()
print("Top 3 Recommended Colleagues")
print("-" * 130)
print(df.loc[recommended_indices].drop(columns=['soup']).to_string(index=False))

Profile of klara_007
----------------------------------------------------------------------------------------------------------------------------------
       id   teams previous_experience                                    hobbies       sports personality_traits
klara_007 team_03           Competent listening to podcasts journaling gardening table tennis creative innovator

Top 3 Recommended Colleagues
----------------------------------------------------------------------------------------------------------------------------------
           id   teams previous_experience                                         hobbies       sports     personality_traits
     emma_042 team_03   Advanced beginner        walking journaling listening to podcasts table tennis   empathetic innovator
christian_049 team_03           Competent listening to podcasts journaling puzzle-solving table tennis cooperative empathetic
    simon_001 team_02              Expert      reading maps walking listening to po